In [1]:
!pip install boto3

^C
ERROR: Operation cancelled by user


In [8]:
import io
import zipfile
import boto3
import pandas as pd

# Connect to MinIO
s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="quantum",
    aws_secret_access_key="quantum-course-only",
    region_name="us-east-1",
)

# 1. Discover or create bucket
existing_buckets = [
    b["Name"] for b in s3.list_buckets().get("Buckets", [])
]
print(f"Available MinIO Buckets: {existing_buckets}")

bucket = "raw"
if bucket not in existing_buckets:
    if existing_buckets:
        bucket = existing_buckets[0]
        print(f"Using existing bucket: {bucket}")
    else:
        s3.create_bucket(Bucket=bucket)
        print(f"Created bucket: {bucket}")

# 2. List objects in bucket
response = s3.list_objects_v2(Bucket=bucket)
objects = response.get("Contents", [])
print(f"\n=== Objects in MinIO Bucket ('{bucket}') ===")
if not objects:
    print(
        "Bucket is empty. Run 'make seed' in your terminal to populate raw data."
    )
else:
    for obj in objects:
        print(f" - {obj['Key']} ({obj['Size']:,} bytes)")

    # 3. Parse Syndrome CSV Row
    syndrome_key = next(
        (o["Key"] for o in objects if "syndromes" in o["Key"].lower()), None
    )
    if syndrome_key:
        obj_syndrome = s3.get_object(Bucket=bucket, Key=syndrome_key)
        with zipfile.ZipFile(
            io.BytesIO(obj_syndrome["Body"].read()), "r"
        ) as z:
            csv_members = [
                m
                for m in z.namelist()
                if m.endswith(".csv") and not m.startswith("__")
            ]
            if csv_members:
                with z.open(csv_members[0]) as f:
                    print("\n=== Syndrome CSV Sample Row ===")
                    print(pd.read_csv(f, nrows=1))

    # 4. Parse Google Surface Code Records
    google_key = next(
        (o["Key"] for o in objects if "google" in o["Key"].lower()), None
    )
    if google_key:
        obj_google = s3.get_object(Bucket=bucket, Key=google_key)
        with zipfile.ZipFile(
            io.BytesIO(obj_google["Body"].read()), "r"
        ) as z:
            b8_members = [m for m in z.namelist() if m.endswith(".b8")]
            if b8_members:
                with z.open(b8_members[0]) as f:
                    first_byte = f.read(1)
                    bits = f"{ord(first_byte):08b}" if first_byte else "EOF"
                    print("\n=== Stim .b8 Binary Sample Byte ===")
                    print(f"File: {b8_members[0]} -> First Byte Bits: {bits}")

            label_members = [m for m in z.namelist() if m.endswith(".01")]
            if label_members:
                with z.open(label_members[0]) as f:
                    first_line = f.readline().decode("utf-8").strip()
                    print("\n=== .01 Label File Sample Line ===")
                    print(f"File: {label_members[0]} -> First Line: {first_line}")

Available MinIO Buckets: ['quantum-lake']
Using existing bucket: quantum-lake

=== Objects in MinIO Bucket ('quantum-lake') ===
 - bronze/source=google_qec/google-surface-code-curated.zip (14,638,673 bytes)
 - bronze/source=qasmbench/qasmbench-qec.zip (144,172 bytes)
 - bronze/source=qec_syndromes/syndromes_dataset.zip (358,017 bytes)
 - metadata/course-release/ATTRIBUTION.md (1,387 bytes)
 - metadata/course-release/bundle-manifest.json (1,349 bytes)

=== Syndrome CSV Sample Row ===
   labels                                          syndromes  quantity
0       0  ((0, 0, 0, 0), (0, 0, 0, 0), (0, 0, 0, 0), (0,...   9987291

=== Stim .b8 Binary Sample Byte ===
File: surface_code_bX_d3_r25_center_3_5/detection_events.b8 -> First Byte Bits: 00000000

=== .01 Label File Sample Line ===
File: surface_code_bX_d3_r25_center_3_5/obs_flips_actual.01 -> First Line: 1


In [11]:
import hashlib
import io
import os
import zipfile
import boto3
import pandas as pd

# 1. Ensure target directories exist
os.makedirs("silver/qec_syndromes", exist_ok=True)
os.makedirs("results/part1", exist_ok=True)

# 2. Connect to MinIO
s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="quantum",
    aws_secret_access_key="quantum-course-only",
)

# 3. Stream Bronze Archive
obj = s3.get_object(
    Bucket="quantum-lake",
    Key="bronze/source=qec_syndromes/syndromes_dataset.zip",
)
raw_bytes = obj["Body"].read()
archive_sha = hashlib.sha256(raw_bytes).hexdigest()

silver_rows = []
trace_rows = []

with zipfile.ZipFile(io.BytesIO(raw_bytes)) as z:
    for member in [m for m in z.namelist() if m.endswith(".csv")]:
        df = pd.read_csv(z.open(member))
        for idx, row in df.iterrows():
            rec_id = f"qec_syn_{hashlib.md5(f'{member}_{idx}'.encode()).hexdigest()[:12]}"

            # Parse string tuple into 16 binary bytes
            tuple_vals = eval(row["syndromes"])
            bit_array = bytes([bit for sub in tuple_vals for bit in sub])

            silver_rows.append(
                {
                    "source_record_id": rec_id,
                    "experiment_id": "syn_sim_p001",
                    "physical_fault_rate": 0.001,
                    "syndrome_bits": bit_array,
                    "round_count": 4,
                    "check_count": 4,
                    "logical_error_label": bool(row["labels"]),
                    "quantity": int(row["quantity"]),
                }
            )

            trace_rows.append(
                {
                    "source_record_id": rec_id,
                    "source_name": "qec_syndromes",
                    "bronze_object": "bronze/source=qec_syndromes/syndromes_dataset.zip",
                    "archive_member": member,
                    "record_locator": f"row_{idx}",
                    "input_sha256": archive_sha,
                }
            )

# 4. Write Parquet Outputs
df_silver = pd.DataFrame(silver_rows)
df_trace = pd.DataFrame(trace_rows)

df_silver.to_parquet("silver/qec_syndromes/syndrome_observation.parquet")
df_trace.to_parquet("results/part1/source_trace.parquet")

print("Successfully wrote:")
print(" - silver/qec_syndromes/syndrome_observation.parquet")
print(" - results/part1/source_trace.parquet")

Successfully wrote:
 - silver/qec_syndromes/syndrome_observation.parquet
 - results/part1/source_trace.parquet


In [12]:
import pandas as pd

# Read back the written Silver table
df_syn = pd.read_parquet("silver/qec_syndromes/syndrome_observation.parquet")
print(f"=== Syndrome Observation Rows: {len(df_syn):,} ===")
print(df_syn.head(2))

# Read back the written Trace table
df_trace = pd.read_parquet("results/part1/source_trace.parquet")
print(f"\n=== Source Trace Rows: {len(df_trace):,} ===")
print(df_trace.head(2))

=== Syndrome Observation Rows: 75,598 ===
       source_record_id experiment_id  physical_fault_rate  \
0  qec_syn_c268447b52d4  syn_sim_p001                0.001   
1  qec_syn_b5e76db40a53  syn_sim_p001                0.001   

                                       syndrome_bits  round_count  \
0  b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...            4   
1  b'\x00\x00\x01\x00\x00\x00\x01\x00\x00\x00\x00...            4   

   check_count  logical_error_label  quantity  
0            4                False   9987291  
1            4                False       486  

=== Source Trace Rows: 75,598 ===
       source_record_id    source_name  \
0  qec_syn_c268447b52d4  qec_syndromes   
1  qec_syn_b5e76db40a53  qec_syndromes   

                                       bronze_object  \
0  bronze/source=qec_syndromes/syndromes_dataset.zip   
1  bronze/source=qec_syndromes/syndromes_dataset.zip   

                archive_member record_locator  \
0  d-3_pfr-0.000010_nb-10M.csv         

In [13]:
import hashlib
import io
import os
import re
import zipfile
import boto3
import pandas as pd

# Ensure output directories exist
os.makedirs("silver/google_qec", exist_ok=True)
os.makedirs("results/part1", exist_ok=True)

s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="quantum",
    aws_secret_access_key="quantum-course-only",
)

# Stream Google QEC Bronze zip
key = "bronze/source=google_qec/google-surface-code-curated.zip"
obj = s3.get_object(Bucket="quantum-lake", Key=key)
raw_bytes = obj["Body"].read()
archive_sha = hashlib.sha256(raw_bytes).hexdigest()

exp_rows = []
shot_rows = []
trace_rows = []
issue_rows = []

with zipfile.ZipFile(io.BytesIO(raw_bytes)) as z:
    all_members = z.namelist()

    # Discover unique experiment directories
    exp_dirs = sorted(
        list(
            {
                m.split("/")[0]
                for m in all_members
                if "/" in m and not m.startswith("__")
            }
        )
    )

    for exp_dir in exp_dirs:
        # Parse folder name parameters (e.g. surface_code_bX_d3_r25_center_3_5)
        # Regex to capture: basis, distance, rounds, center_row, center_col
        match = re.search(
            r"surface_code_b([XZ])_d(\d+)_r(\d+)_center_(\d+)_(\d+)", exp_dir
        )
        if not match:
            continue

        basis, distance, rounds, c_row, c_col = match.groups()
        exp_id = f"google_{exp_dir}"
        exp_rec_id = (
            f"exp_{hashlib.md5(exp_id.encode()).hexdigest()[:12]}"
        )

        b8_member = f"{exp_dir}/detection_events.b8"
        label_member = f"{exp_dir}/obs_flips_actual.01"

        if b8_member not in all_members or label_member not in all_members:
            issue_rows.append(
                {
                    "source_name": "google_qec",
                    "bronze_object": key,
                    "archive_member": exp_dir,
                    "issue_type": "MISSING_COMPANION_FILE",
                    "details": "Missing detection_events.b8 or obs_flips_actual.01",
                }
            )
            continue

        # Read companion files
        b8_bytes = z.read(b8_member)
        label_lines = [
            line.strip()
            for line in z.read(label_member).decode("utf-8").splitlines()
            if line.strip()
        ]

        num_shots = len(label_lines)
        bytes_per_shot = (
            len(b8_bytes) // num_shots if num_shots > 0 else 0
        )

        # Validate alignment
        if len(b8_bytes) != bytes_per_shot * num_shots:
            issue_rows.append(
                {
                    "source_name": "google_qec",
                    "bronze_object": key,
                    "archive_member": exp_dir,
                    "issue_type": "ALIGNMENT_MISMATCH",
                    "details": f"b8 byte length {len(b8_bytes)} not divisible by shot count {num_shots}",
                }
            )

        # Add Experiment metadata record
        exp_rows.append(
            {
                "source_record_id": exp_rec_id,
                "experiment_id": exp_id,
                "basis": basis,
                "distance": int(distance),
                "rounds": int(rounds),
                "shots": num_shots,
                "center_row": int(c_row),
                "center_col": int(c_col),
                "measurement_count": bytes_per_shot * 8,
                "detector_count": bytes_per_shot * 8,
            }
        )

        trace_rows.append(
            {
                "source_record_id": exp_rec_id,
                "source_name": "google_qec",
                "bronze_object": key,
                "archive_member": exp_dir,
                "record_locator": "experiment_metadata",
                "input_sha256": archive_sha,
            }
        )

        # Process individual aligned shots
        for shot_idx in range(num_shots):
            shot_rec_id = f"google_shot_{hashlib.md5(f'{exp_id}_{shot_idx}'.encode()).hexdigest()[:12]}"
            start_b = shot_idx * bytes_per_shot
            shot_b8 = b8_bytes[start_b : start_b + bytes_per_shot]

            # Count set bits in detector bytes
            event_cnt = bin(
                int.from_bytes(shot_b8, byteorder="little")
            ).count("1")

            shot_rows.append(
                {
                    "source_record_id": shot_rec_id,
                    "experiment_id": exp_id,
                    "shot_index": shot_idx,
                    "measurement_bits": b"",
                    "sweep_bits": b"",
                    "detector_bits": shot_b8,
                    "detector_event_count": event_cnt,
                    "actual_observable_flip": bool(
                        int(label_lines[shot_idx])
                    ),
                    "belief_matching_prediction": False,
                    "correlated_matching_prediction": False,
                    "pymatching_prediction": False,
                    "tensor_network_contraction_prediction": False,
                }
            )

            trace_rows.append(
                {
                    "source_record_id": shot_rec_id,
                    "source_name": "google_qec",
                    "bronze_object": key,
                    "archive_member": f"{b8_member} + {label_member}",
                    "record_locator": f"shot_index_{shot_idx}",
                    "input_sha256": archive_sha,
                }
            )

# Save Google QEC Silver Tables
df_exp = pd.DataFrame(exp_rows)
df_shots = pd.DataFrame(shot_rows)
df_issues = pd.DataFrame(issue_rows)

df_exp.to_parquet("silver/google_qec/experiment.parquet")
df_shots.to_parquet("silver/google_qec/shot.parquet")

# Append new trace rows to existing source_trace.parquet
df_trace_existing = pd.read_parquet("results/part1/source_trace.parquet")
df_trace_updated = pd.concat(
    [df_trace_existing, pd.DataFrame(trace_rows)], ignore_index=True
)
df_trace_updated.to_parquet("results/part1/source_trace.parquet")

if not df_issues.empty:
    df_issues.to_parquet("results/part1/data_issues.parquet")

print("=== Google QEC Silver Pipeline Completed ===")
print(f"Experiments Processed: {len(df_exp):,}")
print(f"Shots Processed: {len(df_shots):,}")
print(f"Total Source Trace Rows: {len(df_trace_updated):,}")
print(f"Quarantined Issues: {len(df_issues):,}")

=== Google QEC Silver Pipeline Completed ===
Experiments Processed: 5
Shots Processed: 250,000
Total Source Trace Rows: 325,603
Quarantined Issues: 0


In [14]:
import hashlib
import io
import os
import re
import zipfile
import boto3
import pandas as pd

# Ensure output directories exist
os.makedirs("silver/qasmbench", exist_ok=True)
os.makedirs("results/part1", exist_ok=True)

s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="quantum",
    aws_secret_access_key="quantum-course-only",
)

# Stream QASMBench Bronze zip
key = "bronze/source=qasmbench/qasmbench-qec.zip"
obj = s3.get_object(Bucket="quantum-lake", Key=key)
raw_bytes = obj["Body"].read()
archive_sha = hashlib.sha256(raw_bytes).hexdigest()

circ_rows = []
check_rows = []
corr_rows = []
trace_rows = []
issue_rows = []

with zipfile.ZipFile(io.BytesIO(raw_bytes)) as z:
    qasm_members = [
        m
        for m in z.namelist()
        if m.endswith(".qasm") and not m.startswith("__")
    ]

    for member in qasm_members:
        try:
            content = z.read(member).decode("utf-8")
            lines = [line.strip() for line in content.splitlines()]

            # Determine identifiers
            parts = member.split("/")
            bench_name = parts[0] if len(parts) > 1 else "qasmbench"
            file_name = os.path.basename(member)
            circ_id = file_name.replace(".qasm", "")

            circ_rec_id = (
                f"qasm_c_{hashlib.md5(member.encode()).hexdigest()[:12]}"
            )

            # 1. Parse Register Declarations and Qubit Counts
            reg_decls = []
            qubit_cnt = 0
            for line in lines:
                if line.startswith("qreg"):
                    reg_decls.append(line)
                    match = re.search(r"\[(\d+)\]", line)
                    if match:
                        qubit_cnt += int(match.group(1))

            # 2. Count Executed Operations
            measure_cnt = sum(1 for l in lines if l.startswith("measure"))
            two_q_cnt = sum(
                1
                for l in lines
                if l.startswith("cx ") or l.startswith("cz ")
            )

            circ_rows.append(
                {
                    "source_record_id": circ_rec_id,
                    "circuit_id": circ_id,
                    "benchmark_name": bench_name,
                    "variant": "source",
                    "register_declarations": " ".join(reg_decls),
                    "qubit_count": qubit_cnt,
                    "measurement_count": measure_cnt,
                    "two_qubit_gate_count": two_q_cnt,
                }
            )

            trace_rows.append(
                {
                    "source_record_id": circ_rec_id,
                    "source_name": "qasmbench",
                    "bronze_object": key,
                    "archive_member": member,
                    "record_locator": "circuit_definition",
                    "input_sha256": archive_sha,
                }
            )

            # 3. Parse Stabilizer Checks and Conditional Corrections
            for line_idx, line in enumerate(lines):
                # Parity / Stabilizer check detection (e.g. cx data[0], ancilla[0]; measure ancilla[0] -> syndrome[0];)
                if line.startswith("measure") and "->" in line:
                    chk_rec_id = f"qasm_chk_{hashlib.md5(f'{member}_{line_idx}'.encode()).hexdigest()[:12]}"
                    m_parts = line.replace("measure ", "").split("->")
                    ancilla = m_parts[0].strip()
                    synd_bit = m_parts[1].replace(";", "").strip()

                    check_rows.append(
                        {
                            "source_record_id": chk_rec_id,
                            "circuit_id": circ_id,
                            "check_id": f"chk_{ancilla}",
                            "ancilla_qubit": ancilla,
                            "data_qubits": [f"data[{i}]" for i in range(2)],
                            "syndrome_bit": synd_bit,
                        }
                    )

                    trace_rows.append(
                        {
                            "source_record_id": chk_rec_id,
                            "source_name": "qasmbench",
                            "bronze_object": key,
                            "archive_member": member,
                            "record_locator": f"line_{line_idx}",
                            "input_sha256": archive_sha,
                        }
                    )

                # Conditional correction detection (e.g. if (syndrome==1) x data[0];)
                elif line.startswith("if"):
                    corr_rec_id = f"qasm_cor_{hashlib.md5(f'{member}_{line_idx}'.encode()).hexdigest()[:12]}"
                    cond_match = re.search(
                        r"if\s*\((.*?)\s*==\s*(\d+)\)\s*(\w+)\s+(.*?);", line
                    )
                    if cond_match:
                        c_reg, c_val, gate, t_qubit = cond_match.groups()
                        corr_rows.append(
                            {
                                "source_record_id": corr_rec_id,
                                "circuit_id": circ_id,
                                "condition_register": c_reg.strip(),
                                "condition_value": int(c_val),
                                "gate": gate.strip(),
                                "target_qubit": t_qubit.strip(),
                            }
                        )

                        trace_rows.append(
                            {
                                "source_record_id": corr_rec_id,
                                "source_name": "qasmbench",
                                "bronze_object": key,
                                "archive_member": member,
                                "record_locator": f"line_{line_idx}",
                                "input_sha256": archive_sha,
                            }
                        )

        except Exception as e:
            issue_rows.append(
                {
                    "source_name": "qasmbench",
                    "bronze_object": key,
                    "archive_member": member,
                    "issue_type": "PARSE_ERROR",
                    "details": str(e),
                }
            )

# Save QASMBench Silver Tables
df_circ = pd.DataFrame(circ_rows)
df_check = pd.DataFrame(check_rows)
df_corr = pd.DataFrame(corr_rows)

df_circ.to_parquet("silver/qasmbench/circuit.parquet")
df_check.to_parquet("silver/qasmbench/stabilizer_check.parquet")
df_corr.to_parquet("silver/qasmbench/conditional_correction.parquet")

# Update central source trace
df_trace_existing = pd.read_parquet("results/part1/source_trace.parquet")
df_trace_final = pd.concat(
    [df_trace_existing, pd.DataFrame(trace_rows)], ignore_index=True
)
df_trace_final.to_parquet("results/part1/source_trace.parquet")

# Update issues log if needed
if issue_rows:
    df_issues = pd.DataFrame(issue_rows)
    df_issues.to_parquet("results/part1/data_issues.parquet")

print("=== QASMBench Silver Pipeline Completed ===")
print(f"Circuits Parsed: {len(df_circ):,}")
print(f"Stabilizer Checks Extracted: {len(df_check):,}")
print(f"Conditional Corrections Extracted: {len(df_corr):,}")
print(f"Final Central Trace Rows: {len(df_trace_final):,}")

=== QASMBench Silver Pipeline Completed ===
Circuits Parsed: 6
Stabilizer Checks Extracted: 27
Conditional Corrections Extracted: 6
Final Central Trace Rows: 325,642


In [15]:
import os
import pandas as pd

silver_tables = [
    "silver/qec_syndromes/syndrome_observation.parquet",
    "silver/google_qec/experiment.parquet",
    "silver/google_qec/shot.parquet",
    "silver/qasmbench/circuit.parquet",
    "silver/qasmbench/stabilizer_check.parquet",
    "silver/qasmbench/conditional_correction.parquet",
]

print("=== Silver Layer Verification Audit ===")
total_silver_rows = 0

for table_path in silver_tables:
    exists = os.path.exists(table_path)
    if exists:
        df = pd.read_parquet(table_path)
        row_cnt = len(df)
        has_id = "source_record_id" in df.columns
        total_silver_rows += row_cnt
        print(
            f"✓ {table_path:<50} | Rows: {row_cnt:>7,} | Has ID: {has_id}"
        )
    else:
        print(f"✗ {table_path:<50} | MISSING")

# Trace file audit
df_trace = pd.read_parquet("results/part1/source_trace.parquet")
print(
    f"\nCentral Trace Audit File: {len(df_trace):,} records logged."
)
print(
    f"Trace Link Integrity: {'PASSED' if len(df_trace) == total_silver_rows else 'FAILED'}"
)

=== Silver Layer Verification Audit ===
✓ silver/qec_syndromes/syndrome_observation.parquet  | Rows:  75,598 | Has ID: True
✓ silver/google_qec/experiment.parquet               | Rows:       5 | Has ID: True
✓ silver/google_qec/shot.parquet                     | Rows: 250,000 | Has ID: True
✓ silver/qasmbench/circuit.parquet                   | Rows:       6 | Has ID: True
✓ silver/qasmbench/stabilizer_check.parquet          | Rows:      27 | Has ID: True
✓ silver/qasmbench/conditional_correction.parquet    | Rows:       6 | Has ID: True

Central Trace Audit File: 325,642 records logged.
Trace Link Integrity: PASSED


In [16]:
import pandas as pd

# 1. Load any Silver table
df = pd.read_parquet("silver/qasmbench/circuit.parquet")

# 2. Display basic summary info
print("=== Table Shape (Rows, Columns) ===")
print(df.shape)

print("\n=== Column Types ===")
print(df.dtypes)

print("\n=== First 5 Rows ===")
print(df.head())

=== Table Shape (Rows, Columns) ===
(6, 8)

=== Column Types ===
source_record_id         object
circuit_id               object
benchmark_name           object
variant                  object
register_declarations    object
qubit_count               int64
measurement_count         int64
two_qubit_gate_count      int64
dtype: object

=== First 5 Rows ===
      source_record_id                        circuit_id benchmark_name  \
0  qasm_c_0f71d0f1846a             error_correctiond3_n5          small   
1  qasm_c_bffc66a0438d  error_correctiond3_n5_transpiled          small   
2  qasm_c_bc7a4f0002dc                         qec_en_n5          small   
3  qasm_c_8df8b39f4057              qec_en_n5_transpiled          small   
4  qasm_c_759f850ff8db                         qec_sm_n5          small   

  variant  register_declarations  qubit_count  measurement_count  \
0  source             qreg q[5];            5                  5   
1  source             qreg q[5];            5           

In [18]:
# 1. Load any Silver table
df = pd.read_parquet("silver/google_qec/experiment.parquet")

# 2. Display basic summary info
print("=== Table Shape (Rows, Columns) ===")
print(df.shape)

print("\n=== Column Types ===")
print(df.dtypes)

print("\n=== First 5 Rows ===")
print(df.head())

=== Table Shape (Rows, Columns) ===
(5, 10)

=== Column Types ===
source_record_id     object
experiment_id        object
basis                object
distance              int64
rounds                int64
shots                 int64
center_row            int64
center_col            int64
measurement_count     int64
detector_count        int64
dtype: object

=== First 5 Rows ===
   source_record_id                             experiment_id basis  distance  \
0  exp_dc1c8c81dc52  google_surface_code_bX_d3_r25_center_3_5     X         3   
1  exp_b3206f3b7bf7  google_surface_code_bX_d3_r25_center_5_3     X         3   
2  exp_2cee3fad4f1c  google_surface_code_bX_d3_r25_center_5_7     X         3   
3  exp_a644e3a2d8d6  google_surface_code_bX_d3_r25_center_7_5     X         3   
4  exp_fbfa44cc439a  google_surface_code_bX_d5_r25_center_5_5     X         5   

   rounds  shots  center_row  center_col  measurement_count  detector_count  
0      25  50000           3           5             

In [19]:
# 1. Load any Silver table
df = pd.read_parquet("silver/qec_syndromes/syndrome_observation.parquet")

# 2. Display basic summary info
print("=== Table Shape (Rows, Columns) ===")
print(df.shape)

print("\n=== Column Types ===")
print(df.dtypes)

print("\n=== First 5 Rows ===")
print(df.head())

=== Table Shape (Rows, Columns) ===
(75598, 8)

=== Column Types ===
source_record_id        object
experiment_id           object
physical_fault_rate    float64
syndrome_bits           object
round_count              int64
check_count              int64
logical_error_label       bool
quantity                 int64
dtype: object

=== First 5 Rows ===
       source_record_id experiment_id  physical_fault_rate  \
0  qec_syn_c268447b52d4  syn_sim_p001                0.001   
1  qec_syn_b5e76db40a53  syn_sim_p001                0.001   
2  qec_syn_b6b73a9d9e86  syn_sim_p001                0.001   
3  qec_syn_f5b65dcca5cc  syn_sim_p001                0.001   
4  qec_syn_bc0167551f46  syn_sim_p001                0.001   

                                       syndrome_bits  round_count  \
0  b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...            4   
1  b'\x00\x00\x01\x00\x00\x00\x01\x00\x00\x00\x00...            4   
2  b'\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x00...            4  